# Poker McPokerface — Play

An interactive 5-card draw notebook where you sit at a table with three MockBots.

**MockBots are stand-ins** that respond with hard-coded JSON instead of calling a real LLM — they let us exercise the engine before any models are downloaded. Once `bots/ollama_bot.py` lands, swapping a MockBot for a real LLM bot is a one-line change in the cell that builds the table.

## How to play

Run the cells top to bottom. The final cell is a hand loop — when it's your turn the table renders inline (HTML in Jupyter, ASCII otherwise) and you type your action into the prompt that appears.

Legal action inputs:

- `fold` / `f`
- `check` / `call` / `c`  (engine treats one as the other when one is illegal)
- `raise 25` / `r 25` / `raise to 50`
- `all-in` / `allin` / `shove`
- empty (just press Enter) — checks if free, otherwise re-prompts

Discard input (during the draw phase): comma-separated indices `0,2,4`, or `all`, or empty for stand-pat.

## 1. Imports + path setup

In [1]:
import sys
from pathlib import Path
# Notebooks live in `notebooks/`. Make sure the project root is on
# sys.path so we can `import engine`, `import bots`, etc.
_here = Path.cwd()
_root = _here.parent if _here.name == 'notebooks' else _here
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print(f'Project root: {_root}')

Project root: C:\Users\35387\Masters\Semester 2\GenAI\Poker


In [4]:
from engine import Player, Seat, Game
from bots import MockBot, Personality
from ui import HumanAgent

## 2. Define some personalities

Personalities are pure data: an id (used for analytics groupby), a description, and a system prompt that real LLM bots would receive. MockBots ignore the prompt but still carry the id, which keeps the data model consistent for when we swap them out.

In [5]:
aggressive = Personality(
    id='aggressive',
    description='Raises and bluffs frequently.',
    system_prompt='You are an aggressive 5-card draw poker player. Raise often and put pressure on opponents.',
)
cautious = Personality(
    id='cautious',
    description='Calls and checks rather than raising.',
    system_prompt='You are a cautious 5-card draw poker player. Prefer calling and checking; raise only with a strong hand.',
)
tight = Personality(
    id='tight',
    description='Folds whenever there is anything to call.',
    system_prompt='You are a very tight 5-card draw poker player. You only stay in with the strongest hands.',
)

## 3. Build the table

Three MockBots with distinct play styles plus you. MockBot responses are canned JSON strings; the parser reads them just like a real LLM response.

In [6]:
# Canned JSON each MockBot will return for every action / discard call.
raiser_resp  = '{"reasoning": "Always pressure.", "action": "raise", "amount": 5}'
caller_resp  = '{"reasoning": "Just keeping pace.", "action": "call"}'
folder_resp  = '{"reasoning": "Not worth it.", "action": "fold"}'
stand_pat    = '{"reasoning": "Standing pat.", "discards": []}'
draw_two     = '{"reasoning": "Drawing to a hand.", "discards": [3, 4]}'

alice_bot = MockBot('Alice', aggressive, raiser_resp, draw_two,  model_id='mock-aggressive')
bob_bot   = MockBot('Bob',   cautious,   caller_resp, stand_pat, model_id='mock-cautious')
carol_bot = MockBot('Carol', tight,      folder_resp, stand_pat, model_id='mock-tight')

human = HumanAgent(name='You')

seats = [
    Seat(Player('Alice', chips=200), alice_bot),
    Seat(Player('Bob',   chips=200), bob_bot),
    Seat(Player('Carol', chips=200), carol_bot),
    Seat(Player('You',   chips=200), human),
]
game = Game(seats, ante=2, min_bet=5, seed=42)
print('Table ready. Run the next cell to play a hand.')

Table ready. Run the next cell to play a hand.


## 4. Play a hand

Re-run this cell as many times as you like. Each run plays one full hand — antes, deal, betting round, draw, betting round, showdown. Chips persist between cell runs because the same `game` object is reused.

In [ ]:
from ui import render_showdown_html, render_showdown_text
from IPython.display import HTML, display

summary = game.play_hand()

# Render showdown — every player's cards revealed (folded too)
# so you can see what beat you and what folded out. Pretty HTML
# in Jupyter, ASCII fallback otherwise.
try:
    display(HTML(render_showdown_html(summary, reveal_folded=True)))
except Exception:
    print(render_showdown_text(summary, reveal_folded=True))

## 5. Quick sanity-check

Total chips on the table should equal `200 × 4 = 800` (no rebuys are happening here — chips just move between seats).

In [ ]:
total = sum(p.chips for p in game.players)
print(f'Total chips on the table: {total}  (should be 800)')
assert total == 800, 'chip conservation failed!'